**Installing dependencies**

In [119]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from wordcloud import WordCloud

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
!pip install krippendorff

In [ ]:
!pip install tensorflow

**Loading dataset**

In [ ]:
!unzip /content/NLP_Law_Dataset.zip -d /content/NLP_Law


In [ ]:
!ls /content/NLP_Law


In [ ]:
sns.set(style="whitegrid", palette="pastel")
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12

data_path = "/content/NLP_Law"

In [ ]:
files = [
    "acceleration.csv",
    "confidentiality.csv",
    "termination.csv",
    "payment.csv",
    "governing-law.csv",
    "indemnification.csv",
    "force-majeure.csv",
    "licenses.csv",
    "voting.csv",
    "litigation.csv",
    "waiver.csv",
    "vacation.csv",
    "security-deposit.csv",
    "bonus.csv",
    "rent.csv",
    "judgments.csv",
    "survival.csv",
    "non-discrimination.csv",
    "power-of-attorney.csv",
    "employee-benefits.csv"

]

In [ ]:

datasets = {}
for file in files:
    name = file.replace(".csv", "").replace("-", "_")
    datasets[name] = pd.read_csv(os.path.join(data_path, file))

for name, df in datasets.items():
    print(f"{name}")
    print(df.head(), "\n")


**Visualising datasets**

In [ ]:
plt.figure(figsize=(10,6))
for name, df in datasets.items():
    df["clause_length"] = df["clause_text"].apply(lambda x: len(str(x).split()))
    sns.kdeplot(df["clause_length"], label=name, fill=True, alpha=0.4)
plt.title("Clause Length Distribution Across Legal Clause Types")
plt.xlabel("Words per Clause")
plt.ylabel("Density")
plt.legend(title="Clause Type")
plt.show()


In [ ]:
counts = {name: len(df) for name, df in datasets.items()}
sns.barplot(x=list(counts.keys()), y=list(counts.values()), hue=list(counts.keys()),
            palette="pastel", legend=False)
plt.title("Number of Clauses per Legal Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


In [ ]:
colormaps = {
    "acceleration": "Blues",
    "confidentiality": "Greens",
    "termination": "Oranges",
    "payment": "Reds",
    "governing_law": "Purples",
    "indemnification": "pink_r",
    "force_majeure": "cool",
    "licenses": "Set3",
    "voting": "Pastel1",
    "litigation": "Pastel2",
    "waiver": "Wistia",
    "vacation": "spring",
    "security_deposit": "summer",
    "rent": "autumn",
    "judgments": "Pastel2_r",
    "survival": "Pastel1_r",
    "non_discrimination": "Pastel2_r",
    "power_of_attorney": "pink_r",
    "employee_benefits": "coolwarm"
}


In [ ]:
import math

n = len(datasets)
max_cols = 4
rows = math.ceil(n / max_cols)
cols = min(n, max_cols)

fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
axes = axes.flatten()
for i, (name, df) in enumerate(datasets.items()):
    text = " ".join(str(t) for t in df["clause_text"])
    cmap = colormaps.get(name, "coolwarm")
    wc = WordCloud(width=800, height=800, background_color="white", colormap=cmap).generate(text)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].axis("off")
    axes[i].set_title(name.replace("_"," ").title(), fontsize=10)

for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
all_clauses = []

for name, df in datasets.items():
    all_clauses.extend(df["clause_text"].astype(str).tolist())
total_clauses = len(all_clauses)
print(f"Total number of clauses across all datasets: {total_clauses}")


**Examining dataset**

In [ ]:
import pandas as pd

df_details = []

for name, df in datasets.items():
    num_clauses = df.shape[0]
    missing_values = df['clause_text'].isna().sum()
    avg_length = df['clause_text'].dropna().apply(lambda x: len(str(x).split())).mean()
    max_length = df['clause_text'].dropna().apply(lambda x: len(str(x).split())).max()
    min_length = df['clause_text'].dropna().apply(lambda x: len(str(x).split())).min()

    df_details.append({
        'clause_category': name,
        'num_clauses': num_clauses,
        'missing_values': missing_values,
        'avg_length_words': round(avg_length, 2),
        'max_length_words': max_length,
        'min_length_words': min_length
    })
summary = pd.DataFrame(df_details)



In [ ]:
summary

**Pre-processing**

In [ ]:
import re

def remove_html_and_urls(text):
    html_pattern = re.compile(r'<.*?>')
    url_pattern = re.compile(r'http[s]?://\S+|www\.\S+')

    found = html_pattern.search(text) or url_pattern.search(text)
    if found:
        print("HTML or URL found and removed.")

    text = re.sub(html_pattern, '', text)
    text = re.sub(url_pattern, '', text)

    return text

for name, df in datasets.items():
    df['clean_text'] = df['clause_text'].apply(remove_html_and_urls)
    count = df['clause_text'].apply(lambda x: bool(re.search(r'<.*?>|http[s]?://\S+|www\.\S+', x))).sum()
    if count > 0:
        print(f"{count} clauses in {name} contained HTML/URLs and were removed")


In [ ]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

for name, df in datasets.items():
    df['clean_text'] = df['clean_text'].apply(normalize_text)


In [ ]:
def remove_noise(text):
    text = re.sub(r'[^a-zA-Z0-9\s;:]', '', text)
    return text

for name, df in datasets.items():
    df['clean_text'] = df['clean_text'].apply(remove_noise)


In [ ]:
def tokenize_text(text):
    return text.split()




In [ ]:

for name, df in datasets.items():
    df['tokens'] = df['clean_text'].apply(tokenize_text)


In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]


In [ ]:

for name, df in datasets.items():
    df['tokens_lemmatized'] = df['tokens'].apply(lemmatize_tokens)

In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

all_clauses = []
for df in datasets.values():
    all_clauses.extend([' '.join(tokens) for tokens in df['tokens_lemmatized']])

tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(all_clauses)
vocab_size = len(tokenizer.word_index) + 1


In [ ]:
for name, df in datasets.items():
    df['seq'] = tokenizer.texts_to_sequences([' '.join(tokens) for tokens in df['tokens_lemmatized']])


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = max(len(seq) for df in datasets.values() for seq in df['seq'])
for name, df in datasets.items():
    df['seq_padded'] = pad_sequences(df['seq'], maxlen=max_len, padding='post').tolist()


In [ ]:
df = datasets["acceleration"]
print("Original clause text:\n", df['clause_text'].iloc[0])
print("\nLemmatized tokens:\n", df['tokens_lemmatized'].iloc[0])
print("\nInteger sequence:\n", df['seq'].iloc[0])
print("\nPadded sequence:\n", df['seq_padded'].iloc[0])


In [ ]:
import random
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

rows = []
for name, df in datasets.items():
    for idx, row in df.iterrows():
        rows.append((row['seq_padded'], name))

clauses, labels = zip(*rows)
clauses = np.array(clauses)
labels = np.array(labels)


In [ ]:

label_to_indices = {}
for i, lab in enumerate(labels):
    label_to_indices.setdefault(lab, []).append(i)

pairs_a = []
pairs_b = []
pair_labels = []

n_neg_per_pos = 1
sample_limit = len(clauses)


In [ ]:

for i in range(sample_limit):
    a = clauses[i]
    lab = labels[i]
    pos_choices = label_to_indices[lab].copy()
    pos_choices = [p for p in pos_choices if p != i]
    if pos_choices:
        j = random.choice(pos_choices)
        pairs_a.append(a)
        pairs_b.append(clauses[j])
        pair_labels.append(1)
    for _ in range(n_neg_per_pos):
        neg_lab = random.choice([l for l in label_to_indices.keys() if l != lab])
        j = random.choice(label_to_indices[neg_lab])
        pairs_a.append(a)
        pairs_b.append(clauses[j])
        pair_labels.append(0)


In [ ]:

pairs_a = np.array(pairs_a)
pairs_b = np.array(pairs_b)
pair_labels = np.array(pair_labels)

x1_train, x1_val, x2_train, x2_val, y_train, y_val = train_test_split(
    pairs_a, pairs_b, pair_labels, test_size=0.2, random_state=42, stratify=pair_labels)

batch_size = 64

train_ds = tf.data.Dataset.from_tensor_slices(((x1_train.astype(np.int32), x2_train.astype(np.int32)), y_train.astype(np.int32))).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices(((x1_val.astype(np.int32), x2_val.astype(np.int32)), y_val.astype(np.int32))).batch(batch_size).prefetch(tf.data.AUTOTUNE)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

vocab_size = max(int(np.max(clauses)), 1) + 1
embedding_dim = 128
seq_len = clauses.shape[1]

def make_encoder():
    inp = layers.Input(shape=(seq_len,), dtype='int32')
    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True)(inp)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.2))(x)
    x = layers.Bidirectional(layers.LSTM(32, dropout=0.2))(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    return Model(inp, x)

encoder = make_encoder()

In [ ]:
a_in = layers.Input(shape=(seq_len,), dtype='int32')
b_in = layers.Input(shape=(seq_len,), dtype='int32')

a_vec = encoder(a_in)
b_vec = encoder(b_in)

diff = layers.Lambda(lambda tensors: tf.abs(tensors[0] - tensors[1]))([a_vec, b_vec])
merged = layers.Dense(64, activation='relu')(diff)
merged = layers.Dropout(0.3)(merged)
out = layers.Dense(1, activation='sigmoid')(merged)

siamese_model = Model([a_in, b_in], out)
siamese_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'], run_eagerly=True)
siamese_model.summary()

**Training**

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True, verbose=1)
ckpt = ModelCheckpoint('best_siamese.h5', monitor='val_loss', save_best_only=True, verbose=1)

history = siamese_model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[es, ckpt], verbose=1)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid', font_scale=1.1)
palette = ['#CBAACB', '#FFB7B2', '#FFDAC1', '#E2F0CB', '#B5EAD7']  # lilac, blush, sand, mint, sage

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train Loss', color=palette[0], linewidth=2.5)
plt.plot(history.history['val_loss'], label='Val Loss', color=palette[1], linewidth=2.5)
plt.title('Loss over Epochs', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(frameon=False)



In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color=palette[3], linewidth=2.5)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color=palette[4], linewidth=2.5)
plt.title('Accuracy over Epochs', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, accuracy_score, precision_recall_fscore_support
from tensorflow.keras import layers, Model

def make_encoder(seq_len, vocab_size, embedding_dim=128):
    inp = layers.Input(shape=(seq_len,), dtype='int32')
    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True)(inp)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.2))(x)
    x = layers.Bidirectional(layers.LSTM(32, dropout=0.2))(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    return Model(inp, x)

def abs_diff(tensors):
    return tf.abs(tensors[0] - tensors[1])


In [ ]:

seq_len = x1_val.shape[1]
vocab_size = int(np.max(np.concatenate([x1_val, x2_val]))) + 1
encoder = make_encoder(seq_len, vocab_size)

a_in = layers.Input(shape=(seq_len,), dtype='int32')
b_in = layers.Input(shape=(seq_len,), dtype='int32')

a_vec = encoder(a_in)
b_vec = encoder(b_in)

diff = layers.Lambda(abs_diff)([a_vec, b_vec])
merged = layers.Dense(64, activation='relu')(diff)
merged = layers.Dropout(0.3)(merged)
out = layers.Dense(1, activation='sigmoid')(merged)

siamese_model = Model([a_in, b_in], out)
siamese_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

siamese_model.load_weights('/content/best_siamese.h5')

y_prob = siamese_model.predict((x1_val, x2_val), batch_size=64).ravel()
y_pred_bilstm = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_val, y_pred_bilstm)
prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred_bilstm, average='binary', zero_division=0)
auc = roc_auc_score(y_val, y_prob)

print(f'accuracy: {acc:.4f}')
print(f'precision: {prec:.4f}')
print(f'recall: {rec:.4f}')
print(f'f1: {f1:.4f}')
print(f'roc_auc: {auc:.4f}')



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_val, y_pred_bilstm)


In [ ]:
cm

In [ ]:
classification_report(y_val, y_pred_bilstm, zero_division=0)


In [ ]:

plt.figure(figsize=(8,6))
sns.set(style='whitegrid')
sns.heatmap(cm, annot=True, fmt='d', cmap=sns.color_palette("pastel", as_cmap=True),
            cbar=False, linewidths=0.5, linecolor='gray')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix', fontsize=14)
plt.show()


In [ ]:
import pandas as pd
import numpy as np

results = pd.DataFrame({
    'clause_a': [' '.join(map(str, seq)) for seq in x1_val],
    'clause_b': [' '.join(map(str, seq)) for seq in x2_val],
    'true_label': y_val,
    'predicted_label': y_pred_bilstm,
    'probability': y_prob
})

results['is_correct'] = results['true_label'] == results['predicted_label']

correct = results[results['is_correct']].sample(10, random_state=42)
wrong = results[~results['is_correct']].sample(10, random_state=42)


In [ ]:
correct_df = correct.copy()
correct_df['clause_a'] = correct_df['clause_a'].str.slice(0, 100) + '...'
correct_df['clause_b'] = correct_df['clause_b'].str.slice(0, 100) + '...'

display(correct_df[['clause_a', 'clause_b', 'true_label', 'predicted_label', 'probability']])


In [ ]:
wrong_df = wrong.copy()
wrong_df['clause_a'] = wrong_df['clause_a'].str.slice(0, 100) + '...'
wrong_df['clause_b'] = wrong_df['clause_b'].str.slice(0, 100) + '...'

display(wrong_df[['clause_a', 'clause_b', 'true_label', 'predicted_label', 'probability']])


**Attention Based Encoder**

In [ ]:

seq_len = x1_train.shape[1]
embedding_dim = 100
vocab_size = int(np.max(np.concatenate([x1_train, x2_train]))) + 1
num_classes = 2


In [ ]:
inputs = Input(shape=(seq_len,))
x = layers.Embedding(vocab_size, embedding_dim, input_length=seq_len)(inputs)

attn_scores = layers.Dense(1, activation='tanh')(x)
attn_weights = layers.Softmax(axis=1)(attn_scores)
context = layers.Dot(axes=1)([attn_weights, x])
x = layers.Flatten()(context)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model_attn = Model(inputs, outputs)
model_attn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)


In [ ]:
from tensorflow.keras import Input
from sklearn.metrics import f1_score


In [ ]:
history_attn = model_attn.fit(
    x1_train, y_train,
    validation_data=(x1_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

y_prob_attn = model_attn.predict(x1_val, batch_size=64).ravel()
y_pred_attn = (y_prob_attn >= 0.5).astype(int)

print("accuracy:", accuracy_score(y_val, y_pred_attn))
print("f1:", f1_score(y_val, y_pred_attn, average='weighted'))
print(classification_report(y_val, y_pred_attn, zero_division=0))


In [ ]:

print("accuracy:", accuracy_score(y_val, y_pred_attn))
print("f1:", f1_score(y_val, y_pred_attn, average='weighted'))


In [ ]:
classification_report(y_val, y_pred_attn, zero_division=0)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
palette = sns.color_palette('pastel')

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history_attn.history['loss'], label='train loss', color=palette[0], linewidth=2)
plt.plot(history_attn.history['val_loss'], label='val loss', color=palette[1], linewidth=2)
plt.title('loss over epochs', fontsize=12)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()

In [ ]:


plt.subplot(1,2,1)
plt.plot(history_attn.history['accuracy'], label='train acc', color=palette[2], linewidth=2)
plt.plot(history_attn.history['val_accuracy'], label='val acc', color=palette[3], linewidth=2)
plt.title('accuracy over epochs', fontsize=12)
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
model_attn.save('best_attn_model.keras')


In [ ]:
best = tf.keras.models.load_model('best_attn_model.keras')

y_prob = best.predict(x1_val, batch_size=64).ravel()
y_pred_attn = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_val, y_pred_attn)
prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred_attn, average='binary', zero_division=0)
auc = roc_auc_score(y_val, y_prob)


In [ ]:

print(f'accuracy: {acc:.4f}')
print(f'precision: {prec:.4f}')
print(f'recall: {rec:.4f}')
print(f'f1: {f1:.4f}')
print(f'roc_auc: {auc:.4f}')


In [ ]:

cm = confusion_matrix(y_val, y_pred_attn)
print("Confusion Matrix:\n", cm)


In [ ]:
classification_report(y_val, y_pred_attn, zero_division=0)


In [ ]:

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap=sns.color_palette("pastel", as_cmap=True), cbar=False)
plt.xlabel('predicted')
plt.ylabel('actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
idxs = np.where(y_val != y_pred_attn)[0][:10]
for idx in idxs:
    print('true', y_val[idx], 'pred', y_pred_attn[idx], 'prob', y_prob[idx])
    print('clause A:', ' '.join([str(x) for x in x1_val[idx][:50]]))
    print('clause B:', ' '.join([str(x) for x in x2_val[idx][:50]]))
    print('\n')


In [ ]:

results = pd.DataFrame({
    'clause_a': [' '.join(map(str, x[:50])) for x in x1_val],
    'clause_b': [' '.join(map(str, x[:50])) for x in x2_val],
    'true_label': y_val,
    'predicted_label': y_pred_attn,
    'probability': y_prob
})

In [ ]:

results['is_correct'] = results['true_label'] == results['predicted_label']

correct = results[results['is_correct']].sample(10, random_state=42)
wrong = results[~results['is_correct']].sample(10, random_state=42)



In [ ]:

print("Correctly Classified Clause Pairs:\n")
display(correct[['clause_a', 'clause_b', 'true_label', 'predicted_label', 'probability']])

In [ ]:
print("\nWrongly Classified Clause Pairs:\n")
display(wrong[['clause_a', 'clause_b', 'true_label', 'predicted_label', 'probability']])

**Comparison**

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(history.history['val_accuracy'], label='BiLSTM val acc', color='#a5c9ca')
plt.plot(history_attn.history['val_accuracy'], label='Attention val acc', color='#e4bad4')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

cm_bilstm = confusion_matrix(y_val, y_pred_bilstm)
cm_attn = confusion_matrix(y_val, y_pred_attn)

sns.heatmap(cm_bilstm, ax=axes[0], cmap='pastel', cbar=False, square=True)
axes[0].set_title('BiLSTM Confusion Matrix')

sns.heatmap(cm_attn, ax=axes[1], cmap='pastel', cbar=False, square=True)
axes[1].set_title('Attention Confusion Matrix')

plt.show()
